# Pre-call term deposit targeting
A reproducible interview case study: public UCI data, cleaning, EDA, conservative leakage boundary, chronological evaluation, ranked outreach and limitations. The entire source is in `analysis.py`; run this notebook from the project directory.

## 1. Business question and data
With a 20% call capacity, can a model prioritize likely term-deposit subscribers? Source: [Moro et al., UCI Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing). Historical Portugal campaigns, 2008–2010. Subscription is not causal uplift.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
from IPython.display import display, Image
from analysis import ROOT, DATA, FEATURES, NUM, CAT, make_pipe, metric, top_k, main
df = pd.read_csv(DATA, sep=";")
assert len(df) == 41188
display(df.head(), df.shape)

## 2. Data quality and cleaning
`unknown` is a literal category, and -1 in `pdays` is a missing-history sentinel. There are repeated rows but no customer ID to identify repeated clients. Keep rows intact; learn preprocessing on training only.

In [ ]:
audit = pd.DataFrame({"dtype":df.dtypes.astype(str), "nulls":df.isna().sum(), "unknown_strings":[int(df[c].eq("unknown").sum()) if df[c].dtype=="object" else 0 for c in df.columns]})
print("Exact duplicate rows:", df.duplicated().sum())
display(audit)

## 3. Exploratory analysis
Observe target imbalance and prior campaign associations; these are descriptive, not causal. The CSV order is temporal, but it lacks a precise timestamp per row.

In [ ]:
display(df.y.value_counts().to_frame("rows")); display(df.assign(subscribed=df.y.eq("yes")).groupby("poutcome").subscribed.agg(["mean","count"]).sort_values("mean",ascending=False))
display(df.assign(subscribed=df.y.eq("yes")).groupby("month").subscribed.agg(["mean","count"]).sort_values("mean",ascending=False))

## 4. Leakage-safe modeling
We exclude `duration`: UCI explicitly warns it is known only after a call. We also conservatively drop current-contact and contemporaneous socioeconomic fields. Order split 70/15/15; the validation PR-AUC selects a model; score the test once. The source script contains the full, independently runnable fitting and plotting pipeline.

In [ ]:
print("Pre-call columns:", FEATURES)
print("Excluded:", sorted(set(df.columns)-set(FEATURES)-{"y"}))
main()

## 5. Evaluation and operating decision
Read the actual generated results. A fixed 20% queue means selecting the highest-scoring fifth **of each batch**, not using an old probability cutoff. Compare PR-AUC to period prevalence and inspect the calibration curve before any use of probability values.

In [ ]:
import json
r=json.loads((ROOT/"outputs"/"metrics.json").read_text())
display(pd.DataFrame({"train":r["splits"]["train"],"validation":r["splits"]["validation"],"test":r["splits"]["test"]}))
display(pd.DataFrame(r["validation"]).T)
print("Chosen:",r["selected_model"],"Test:",r["test"]);print("Fixed validation threshold on test:",r["test_at_validation_threshold"])

In [ ]:
for name in ("01_overview.png", "02_model_quality.png", "03_operating_point.png", "04_decile_lift.png"):
    print(name)
    display(Image(filename=str(ROOT/"outputs"/name),width=850))

## 6. Interpretation and next steps
Holdout prevalence 38.45% versus 5.57% in training, a sharp temporal shift. The 20% queue captures 578/2,376 subscribers, only 1.22x the test base rate. Validation-picked threshold would call roughly half the test slice. A prospective experiment with contact costs, consent rules, time-stamped customer IDs and a randomized holdout is needed before claiming ROI. This is a historical portfolio example, not a production recommendation.